# Deep E-prop: Credit Assignment Across Time and Depth

**Course:** NeuroAI & ML 4 Neuro — Sommersemester 2026  
**Authors:** Simon Peter · Yannick Säckl · Ruchit Kumar Patel  
**Repository:** github.com/simonpeter02/e-prop-in-deep-networks

---

## Abstract

Training recurrent neural networks requires assigning credit to every synaptic weight for its contribution to past outputs — the *credit assignment problem*. The standard solution, **backpropagation through time (BPTT)**, computes exact gradients but is biologically implausible, stores the entire sequence in memory, and cannot run online. **E-prop** (Bellec et al. 2020) is an online approximation that replaces the full recurrent Jacobian with its diagonal (memory O(n³)→O(n²), step-by-step updates) — but it was only ever defined for a *single* recurrent layer.

This notebook first reproduces single-layer e-prop and then extends it to **deep** recurrent networks following **Millidge (2025)**, whose recursion adds a second, cross-layer eligibility trace. The **main result (Experiment 5)** is that deep e-prop assigns credit across **time and depth simultaneously**: on a hierarchical task that needs both, full deep e-prop tracks BPTT for *every* layer and trains the network, while two surgical controls — removing the **depth** path (∂z/∂h = 0) or the **cross-layer temporal** path (∂z/∂zₜ₋₁ = 0) of the top-layer trace εᶻ — each break it.

| Experiment | Question |
|-----------|----------|
| **1 — Single-layer e-prop** | Does e-prop match BPTT on a delayed-recall task (reproduce Bellec 2020)? |
| **2 — Deep e-prop (2 layers)** | Deep-RTRL verification, then deep e-prop vs d=0 vs BPTT — does the approximation hold across a layer boundary? |
| **3 — Depth sweep** | How does gradient alignment with BPTT scale with network depth? |
| **4 — Leaky integrators (α sweep)** | When does the temporal eligibility trace actually matter (e-prop vs d=0 across α)? |
| **5 — Time *and* depth (main result)** | Does deep e-prop assign credit across time and depth at once? full deep e-prop vs ablate-spatial vs ablate-temporal vs BPTT, on a leaky deep RNN doing hierarchical classify-then-count. |

**Why leaky (not vanilla) tanh in Exp 5:** a vanilla tanh RNN's e-prop temporal carry is the diagonal ψ·Wᵢᵢ ≈ 0.005 (negligible), so e-prop collapses onto the d=0 baseline; a leaky unit adds a (1−α) diagonal carry that e-prop captures *exactly*, giving a real temporal trace with memory horizon τ ≈ 1/(1−α).

**How to read this notebook:** Run all cells top to bottom. `results/` is wiped and fully regenerated on every run (so it only ever holds the current run's outputs); architecture diagrams go to `figures/`. The final cell pushes outputs to GitHub. A commented sMNIST appendix at the end is optional (slow).

---
## 1. Setup

Clone the repository, install dependencies, and auto-detect the compute device (GPU preferred).
All subsequent sections depend on this cell.

In [ ]:
# Clone repo (skip if already present), then pull latest changes
import os

REPO_URL = "https://github.com/simonpeter02/e-prop-in-deep-networks.git"
REPO_DIR = "e-prop-in-deep-networks"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
!pip install -q torch numpy matplotlib scipy

In [ ]:
%matplotlib inline
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
import sys, os

# Auto-detect best available device
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Batch size: larger on GPU so each kernel has enough work to saturate the device
BATCH_GPU  = 128
BATCH_CPU  = 32
BATCH_DEFAULT = BATCH_GPU if DEVICE == "cuda" else BATCH_CPU
print(f"Default batch size: {BATCH_DEFAULT}")

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)

In [ ]:
from tasks.store_and_recall  import generate_batch, task_accuracy
from tasks.cue_accumulation  import generate_batch as ca_batch, task_accuracy as ca_acc
from models.vanilla_rnn      import VanillaRNN
from models.leaky_rnn        import LeakyRNN          # promoted module; supports per-neuron alpha
from models.deep_rnn         import DeepRNN
from models.lif_rnn          import LIFNetwork, ALIFNetwork
from learning_rules.eprop         import compute_eprop_gradients, mse_error as sl_mse, xent_error
from learning_rules.eprop         import compute_eprop_leaky_gradients
from learning_rules.bptt          import compute_bptt_gradients, _mse_loss
from learning_rules.deep_eprop    import compute_deep_eprop_gradients, mse_error
from learning_rules.deep_rtrl     import compute_deep_rtrl_gradients
from learning_rules.interface     import apply_gradients
from learning_rules.bptt          import _xent_loss

SEED       = 42
N_PATTERNS = 4
torch.manual_seed(SEED)
np.random.seed(SEED)

from tasks.shd                 import (generate_batch as shd_batch,
                                       task_accuracy as shd_acc,
                                       T_SHD, N_IN_SHD, N_CLASSES as N_CLASSES_SHD)
from models.deep_alif              import DeepALIFNetwork
from learning_rules.deep_eprop_alif import compute_deep_eprop_alif_gradients

print('Imports OK')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MASTER CONFIG — all tunable knobs in one place                         ║
# ║  Defaults are the MINIMUM that shows clean method separation on CPU.     ║
# ║  Values marked ↑ can be increased for publication-quality results.       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Exp 1: single-layer e-prop (store-and-recall) ───────────────────────────
N_REC_SL    = 100
DELAY_SL    = 2
N_STEPS     = 1000      # ↑
LR          = 1e-3
EVAL_EVERY  = 50
delays_sl   = [1, 2, 3, 5, 10, 20, 50]
n_trials_sl = 30        # ↑

# ── Exp 2A: deep-RTRL verification ──────────────────────────────────────────
N_REC_RTRL  = 10
N_REPS      = 30        # ↑

# ── Exp 2B: 2-layer deep e-prop ─────────────────────────────────────────────
N_REC_2L    = 50
delays_2l_grid = [1, 2, 3, 5, 10, 20]
n_trials    = 30        # ↑
N_STEPS_2L  = 1000      # ↑
LR_2L       = 1e-3
EVAL_2L     = 50
delay_2l    = 2

# ── Exp 3: depth sweep ──────────────────────────────────────────────────────
# Cap at 3 for Colab CPU; change to [1,2,3,4,5] for pub-quality ↑
DEPTHS_SWEEP = [1, 2, 3]
DELAYS_SWEEP = [2, 5, 10]
N_REC_DS     = 50
N_TRIALS_DS  = 15       # ↑
N_STEPS_DS   = 600      # ↑
LR_DS        = 1e-3
EVAL_DS      = 30

# ── Exp 4: leaky RNN alpha sweep ─────────────────────────────────────────────
N_REC_LK     = 100
ALPHAS_LK    = [0.05, 0.1, 0.2, 0.5, 1.0]
DELAY_LK     = 5
N_TRIALS_LK  = 30       # ↑
ALPHA_LC     = 0.1      # alpha for learning curves
N_STEPS_LK   = 2000    # ↑
LR_LK        = 5e-3    # learning-curve LR; cosine sweep is LR-free
EVAL_LK      = 50

# ── Exp 5: cue accumulation + leaky RNN ─────────────────────────────────────
N_REC_CA     = 80       # ↑
ALPHA_CA     = 0.02     # slow leak: τ = 1/α = 50 steps; state retains 0.67 at D=20
N_CUES_CA    = 5        # odd → no ties
CUE_DUR_CA   = 1
ICI_CA       = 5        # inter-cue silence (steps)
DELAYS_CA    = [5, 10, 20, 30, 50]
N_TRIALS_CA  = 20       # ↑
N_STEPS_CA   = 800      # ↑ — see note in learning-curves cell
LR_CA        = 3e-3     # SGD; learning curves are indicative only — see cell note
EVAL_CA      = 40
DELAY_LC_CA  = 20       # delay for learning curves

# ── Batch sizes (set from device; see cell above) ────────────────────────────
BATCH_SL  = BATCH_DEFAULT
BATCH_2L  = BATCH_DEFAULT
BATCH_DS  = BATCH_DEFAULT
BATCH_LK  = BATCH_DEFAULT
BATCH_CA  = 128          # explicit: ≥128 reduces gradient noise on this task

# ── Input sizes ──────────────────────────────────────────────────────────────
n_in     = N_PATTERNS + 2   # store-and-recall: patterns + recall + bias
n_in_lk  = N_PATTERNS + 2   # leaky RNN (same task)
n_in_ca  = 5                 # cue-accum: left, right, recall, noise, bias

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Master config loaded.")

# ── Exp 5: Deep ALIF on Spiking Heidelberg Digits (SHD) ──────────────────────
N_REC_SHD      = 256        # neurons per layer
N_LAYERS_SHD   = 2          # number of recurrent layers
ALPHA_SHD      = 0.9        # membrane: fast trace carry ≈ 0.87, 1/e horizon ~8 steps
RHO_SHD        = 0.98       # adaptation: slow trace 1/e horizon ~50 steps ≈ T/2
# beta scales with (1-rho): a_ss = firing_rate/(1-rho) → at 5% firing, threshold shift ≈ v_th/2
BETA_SHD       = 0.02
V_TH_SHD       = 0.1        # spike threshold
GAMMA_SHD      = 0.3        # surrogate gradient peak magnitude
W_IN_SCALE_SHD = 5.0        # SHD is ~1% sparse; scale W_in so neurons fire at init
W_FF_SCALE_SHD = 8.0        # feedforward init drive also needs boosting
BATCH_SHD      = 64
N_STEPS_SHD    = 3000       # ↑ for production quality
LR_SHD         = 1e-3
GRAD_CLIP_SHD  = 1.0
EVAL_SHD       = 100
N_COSINE_SHD   = 50
T_SWEEP_SHD    = [10, 20, 50, 100]
HIDDEN_KEYS_SHD = (
    ['W_in']
    + [f'W_recs.{l}' for l in range(N_LAYERS_SHD)]
    + [f'W_ffs.{l}'  for l in range(N_LAYERS_SHD - 1)]
    + [f'b_recs.{l}' for l in range(N_LAYERS_SHD)]
)


In [ ]:
def save_fig(fig, name, out_dir="results"):
    os.makedirs(out_dir, exist_ok=True)
    fig.savefig(f"{out_dir}/{name}.pdf", bbox_inches='tight')
    fig.savefig(f"{out_dir}/{name}.svg", bbox_inches='tight')
    print(f"Saved {out_dir}/{name}.pdf / .svg")

def bptt_grads_deep(model, inputs, targets, mask, loss_fn=None):
    if loss_fn is None:
        loss_fn = _mse_loss
    for p in model.parameters():
        if p.grad is not None: p.grad.zero_()
    outputs, _ = model(inputs)
    loss_fn(outputs, targets, mask).backward()
    return {k: p.grad.clone() for k, p in model.named_parameters() if p.grad is not None}

def cosine_keys(g1, g2, keys):
    sims = []
    for k in keys:
        if k not in g1 or k not in g2: continue
        v1, v2 = g1[k].flatten(), g2[k].flatten()
        if v1.norm() < 1e-12 or v2.norm() < 1e-12: continue
        sims.append(F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item())
    return float(np.mean(sims)) if sims else float('nan')

def apply_grads_deep(model, grads, lr):
    with torch.no_grad():
        for k, p in model.named_parameters():
            if k in grads: p.data -= lr * grads[k]

print("Helpers defined")

In [ ]:
# ── Clean slate: wipe results/ so each run holds ONLY the current outputs ─────
# (prevents an over-crowded folder of stale files from earlier naming schemes)
import shutil, os
shutil.rmtree("results", ignore_errors=True)
os.makedirs("results", exist_ok=True)
print("Cleared results/ — will be repopulated by this run.")

---
## 2. Background & Theory

### 2.1 The credit assignment problem in RNNs

An RNN with hidden state $h_t \in \mathbb{R}^n$ and recurrent weight matrix $W_\text{rec}$ processes inputs $x_1, \ldots, x_T$ step by step:

$$h_t = \tanh(W_\text{rec}\, h_{t-1} + W_\text{in}\, x_t + b), \qquad o_t = W_\text{out}\, h_t$$

A loss $\mathcal{L}$ is computed over the output sequence. To update $W_\text{rec}[i,j]$ we need $\partial \mathcal{L}/\partial W_\text{rec}[i,j]$, which requires knowing how $h_t[i]$ — and therefore every future output — depended on a synaptic weight applied at every past timestep. This is the *credit assignment problem*.

---

### 2.2 Backpropagation through time (BPTT) — the exact solution

BPTT *unrolls* the RNN into a feedforward graph of depth $T$ and applies standard reverse-mode autodiff. This gives exact gradients but has two drawbacks:

- **Memory:** must store all $T$ hidden states simultaneously — $O(T \cdot n)$ per sequence
- **Online impossibility:** the loss at step $T$ is needed before any parameter update can start

BPTT is our **ground-truth baseline** throughout this notebook.

---

### 2.3 Real-Time Recurrent Learning (RTRL) — exact but expensive

RTRL maintains a *sensitivity matrix* $P_t \in \mathbb{R}^{n \times n^2}$ that tracks how every hidden unit depends on every weight in real time:

$$P_t[k,\,(i,j)] = \frac{\partial h_t[k]}{\partial W_\text{rec}[i,j]}$$

Updated at every step via

$$P_t[k,\,(i,j)] = \sum_m J_t[k,m]\,P_{t-1}[m,\,(i,j)] + \psi_t[k]\,h_{t-1}[j]\,\delta_{ki}$$

where $J_t[k,m] = \psi_t[k]\,W_\text{rec}[k,m]$ is the Jacobian and $\psi_t[i] = 1 - h_t[i]^2$ is the tanh derivative. RTRL is exact but requires $O(n^3)$ memory — impractical for $n \gtrsim 100$.

---

### 2.4 E-prop — the diagonal approximation (Bellec et al. 2020)

E-prop drops all off-diagonal entries of $P_t$: keep only $k = i$ (neuron $i$'s sensitivity to its own weight $W[i,j]$). This gives the *eligibility trace*:

$$\varepsilon_t[i,j] \;=\; \underbrace{\psi_t[i]\,h_{t-1}[j]}_{\text{instantaneous}} + \underbrace{\psi_t[i]\,W_\text{rec}[i,i]}_{\text{carry}} \cdot \varepsilon_{t-1}[i,j]$$

The gradient approximation is then $\partial\mathcal{L}/\partial W[i,j] \approx \sum_t L_t[i]\,\varepsilon_t[i,j]$, where $L_t[i]$ is the *learning signal* (error projected onto the hidden layer).

**Key property of the carry factor for tanh RNNs:**  
With spectral radius 0.9 and $n=100$, $W_\text{rec}[i,i] \approx 0.07$ and $\psi_t[i] \approx 0.07$, giving carry $\approx 0.005$ per step. The eligibility trace forgets the past almost instantly. 

---

### 2.5 The d=0 baseline — no temporal carry

Setting the carry term to zero gives the *d=0* rule:

$$\varepsilon_t^{d=0}[i,j] \;=\; \psi_t[i]\,h_{t-1}[j]$$

If the carry is already negligible (as for tanh), d=0 should perform identically to e-prop. **Our first prediction: e-prop ≈ d=0 for tanh RNNs.**

---

### 2.6 Deep e-prop — extending to L layers (Millidge 2025)

For an $L$-layer network, credit from the output layer must reach parameters in lower layers. Deep e-prop maintains *cross-layer eligibility traces* $\varepsilon^{l_\text{top}, l_\text{src}}_t$ tracking how layer $l_\text{top}$'s hidden state depends on the parameters of layer $l_\text{src}$:

- **Spatial propagation** uses the full feedforward Jacobian $J_t^{ff} = \psi_t^{l_\text{top}} \odot W_{ff}$ (no approximation)
- **Temporal carry** uses the diagonal approximation as in single-layer e-prop

Memory scales as $O(L^2 \cdot B \cdot n^3)$. **Our second prediction: gradient cosine decreases with $L$, because each additional layer introduces one more diagonal approximation in the spatial chain.**

---

### 2.7 Measuring approximation quality: gradient cosine similarity

We compare approximate gradients to BPTT using the *cosine similarity*:

$$\cos(\hat{g},\, g_\text{BPTT}) = \frac{\hat{g} \cdot g_\text{BPTT}}{|\hat{g}|\;|g_\text{BPTT}|}$$

This measures the **direction** of the gradient, ignoring magnitude. A value of 1 means the approximate rule points exactly the right way; 0 means it is orthogonal to the true gradient (useless for learning); negative values actively mislead. We report means over many random seeds on **untrained models** (no dependence on any particular trained solution).

---

### 2.8 The store-and-recall task

A controlled synthetic benchmark for temporal credit assignment:

1. **Cue phase** (1 step): the network sees one of $P$ one-hot patterns
2. **Delay phase** ($D$ steps): blank input — the network must maintain the pattern in its dynamics
3. **Output phase** (1 step): the network must reproduce the pattern; loss is MSE vs. the cue

Total sequence length $T = 1 + D + 1 = D + 2$. Credit for the output loss must be assigned back across the entire delay to the parameters that encoded the cue. Longer delays demand longer-range temporal credit assignment.

---

### 2.9 Leaky integrators — restoring a temporal trace

A *leaky* tanh unit integrates with rate $\alpha \in (0,1]$:

$$h_t = (1-\alpha)\,h_{t-1} + \alpha\,\tanh(W_\text{rec}\,h_{t-1} + W_\text{in}\,x_t + b)$$

Its diagonal Jacobian is $(1-\alpha) + \alpha\,\psi_t[i]\,W_\text{rec}[i,i]$, so the e-prop carry is $\approx (1-\alpha)$ per step. For $\alpha = 0.1$ that is $0.9$ — a memory horizon $\tau \approx 1/(1-\alpha) \approx 10$ steps — versus $\approx 0.005$ for vanilla tanh. The eligibility trace now persists long enough to carry real temporal credit, and e-prop captures the $(1-\alpha)$ carry **exactly** while d=0 discards it. (This is the non-spiking analog of why the original e-prop paper used slow / adaptive spiking neurons.) Experiments 4 and 5 use leaky units for exactly this reason.

**Prediction:** e-prop $\gg$ d=0 once $\alpha < 1$, and the gap grows as $\alpha \to 0$ and as the delay grows.

### 2.10 Architecture and algorithm diagrams

The following diagrams illustrate the key concepts above. They are generated from
`figures/architecture_diagrams.py` and saved as PDF + SVG alongside this notebook.

In [ ]:
import importlib.util as _ilu

_spec = _ilu.spec_from_file_location("arch", "figures/architecture_diagrams.py")
_arch = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_arch)

_diagram_order = [
    (_arch.fig_store_and_recall,        'Fig 1 — Store-and-recall task (cue / delay / output)'),
    (_arch.fig_single_layer,            'Fig 2 — Single-layer tanh RNN architecture'),
    (_arch.fig_eprop_trace,             'Fig 3 — Eligibility trace: e-prop vs d=0 over time'),
    (_arch.fig_credit_assignment,       'Fig 4 — Credit assignment: BPTT vs e-prop vs d=0'),
    (lambda: _arch.fig_deep_rnn(L=3),   'Fig 5 — 3-layer deep RNN architecture'),
    (_arch.fig_deep_eprop_propagation,  'Fig 6 — Cross-layer trace propagation (deep e-prop)'),
    (_arch.fig_deep_credit_assignment,   'Fig 7 — Deep credit assignment: BPTT vs deep e-prop vs d=0'),
]

_saved = []
for _fn, _title in _diagram_order:
    _fig = _fn()
    _fig.suptitle(_title, fontsize=11, fontweight='bold', y=1.01)
    # Save with consistent names
    _name = _title.split('—')[0].strip().lower().replace(' ', '_').replace('/', '_')
    _fig.savefig(f"figures/{_name}.pdf", bbox_inches='tight')
    _fig.savefig(f"figures/{_name}.svg", bbox_inches='tight')
    _saved.append(_name)
    plt.show()

print("Diagrams displayed and saved:"  )
for s in _saved:
    print(f"  figures/{s}.pdf / .svg")

---
## 3. Experiment 1 — Single-layer e-prop (Minimal Viable Result #1)

**Question:** On the simplest possible case (one hidden layer, short delay), do e-prop and d=0 match BPTT in both learning speed and gradient direction?

**Setup:** A single-layer vanilla tanh RNN ($n=100$ hidden units) is trained on the store-and-recall task with delay $D=2$ (total sequence $T=4$ steps). We compare three learning rules:

| Rule | Description |
|------|-------------|
| **BPTT** | Exact gradient via PyTorch autograd — the gold standard |
| **E-prop** | Diagonal approximation of RTRL; carry $\approx 0.005$ per step |
| **d=0** | E-prop with carry set to zero — purely instantaneous |

**Part A — Learning curves:** All three rules trained for 1000 steps at delay=2. If e-prop is a good approximation, all three curves should converge to 100% accuracy at similar speeds.

**Part B — Gradient cosine similarity vs delay:** On untrained models, we measure $\cos(\hat{g}, g_\text{BPTT})$ for delays 1–50. This shows how alignment degrades as the task demands longer-range credit. The key comparison is the gap between the e-prop and d=0 lines — a large gap would mean the carry term matters; a tiny gap means it is negligible.

**Prediction:** Because the carry factor ($\approx 0.005$) is negligible for random-initialised tanh RNNs, e-prop and d=0 should be nearly indistinguishable. Both should achieve high cosine similarity at short delays and degrade gradually as delay increases.

In [ ]:
def apply_grads_sl(model, grads, lr):
    with torch.no_grad():
        model.W_rec.data -= lr * grads['W_rec']
        model.W_in.data  -= lr * grads['W_in']
        model.b_rec.data -= lr * grads['b_rec']
        model.W_out.data -= lr * grads['W_out']
        model.b_out.data -= lr * grads['b_out']

def train_sl(label, use_bptt=False, d_zero=False, delay=DELAY_SL):
    torch.manual_seed(SEED)
    model = VanillaRNN(n_in, N_REC_SL, N_PATTERNS).to(DEVICE)
    accs = []
    for step in range(N_STEPS):
        inputs, targets, mask = generate_batch(BATCH_SL, N_PATTERNS, delay, 1, 1, DEVICE)
        if use_bptt:
            grads = compute_bptt_gradients(model, inputs, targets, mask)
        else:
            grads = compute_eprop_gradients(model, inputs, targets, mask, sl_mse, d_zero=d_zero)
        apply_grads_sl(model, grads, LR)
        if step % EVAL_EVERY == 0:
            with torch.no_grad():
                out, _ = model(inputs)
            acc = task_accuracy(out, targets, mask)
            accs.append(acc)
            if step % 200 == 0:
                print(f"  [{label}] step {step:4d}  acc={acc:.3f}")
    return accs

print("Training single-layer models...")
accs_eprop = train_sl("e-prop")
accs_d0    = train_sl("d=0",   d_zero=True)
accs_bptt  = train_sl("BPTT",  use_bptt=True)
print("Done.")

In [ ]:
steps = list(range(0, N_STEPS, EVAL_EVERY))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, accs_eprop, label='e-prop',  marker='o', markersize=3)
ax.plot(steps, accs_d0,    label='d=0',     marker='s', markersize=3)
ax.plot(steps, accs_bptt,  label='BPTT',    marker='^', markersize=3)
ax.axhline(1/N_PATTERNS, color='gray', linestyle='--', label='chance')
ax.set_xlabel('Training step'); ax.set_ylabel('Accuracy')
ax.set_title(f'Store-and-recall learning curves (single layer, delay={DELAY_SL})')
ax.legend(); fig.tight_layout()
save_fig(fig, 'exp1_tanh_store_recall_delay2_learning_curves')
plt.show()

In [ ]:
delays_sl   = [1, 2, 3, 5, 10, 20, 50]
n_trials_sl = 30

cos_eprop, cos_d0 = [], []
for delay in delays_sl:
    model = VanillaRNN(n_in, N_REC_SL, N_PATTERNS).to(DEVICE)
    se, sd = [], []
    for _ in range(n_trials_sl):
        inp, tgt, msk = generate_batch(BATCH_SL, N_PATTERNS, delay, 1, 1, DEVICE)
        gb = compute_bptt_gradients(model, inp, tgt, msk)
        ge = compute_eprop_gradients(model, inp, tgt, msk, sl_mse, d_zero=False)
        gd = compute_eprop_gradients(model, inp, tgt, msk, sl_mse, d_zero=True)
        def c(g1, g2):
            v1 = torch.cat([g1[k].flatten() for k in ['W_rec','W_in','b_rec']])
            v2 = torch.cat([g2[k].flatten() for k in ['W_rec','W_in','b_rec']])
            return F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
        se.append(c(ge, gb)); sd.append(c(gd, gb))
    cos_eprop.append(np.mean(se)); cos_d0.append(np.mean(sd))
    print(f"  delay={delay:3d}  cos(e-prop)={cos_eprop[-1]:.4f}  cos(d=0)={cos_d0[-1]:.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(delays_sl, cos_eprop, label='e-prop vs BPTT', marker='o')
ax.plot(delays_sl, cos_d0,    label='d=0 vs BPTT',    marker='s')
ax.axhline(0, color='gray', linestyle='--')
ax.set_xlabel('Delay (steps)'); ax.set_ylabel('Gradient cosine similarity')
ax.set_title('Gradient alignment vs delay — single-layer RNN')
ax.legend(); fig.tight_layout()
save_fig(fig, 'exp1_tanh_store_recall_cosine_vs_delay')
plt.show()

---
## 4. Experiment 2A — Deep-RTRL verification (Minimal Viable Result #2)

**Question:** Is our deep e-prop implementation computing the right gradients?

Before running approximate methods, we need a ground-truth reference for deep networks. **Deep-RTRL** is the exact version of RTRL extended to $L$ layers — it maintains the full $O(n^4)$ sensitivity tensor and is therefore only feasible at tiny $n$. We run it at $n=10$ and compare its gradients to BPTT on 30 random seeds.

**What we check:** The *direction error* $\|\hat{g}/|\hat{g}| - g_\text{BPTT}/|g_\text{BPTT}|\|_2$ should be < $10^{-4}$ (numerical precision). A pass here confirms that:
1. Our PyTorch implementation of the deep-RTRL recursion is correct
2. Deep-RTRL and BPTT compute mathematically identical gradients (up to floating-point error)
3. Any deviation seen in later e-prop experiments is due to the diagonal approximation, not bugs

> **Note:** Deep-RTRL is intentionally run on CPU — the tensors are tiny ($n=10$, $O(n^4) = 10^4$ elements) and the computation is not worth the GPU dispatch overhead.

In [ ]:
N_REC_RTRL = 10
N_REPS     = 30

max_dir_err = 0.0
worst_key   = None

for rep in range(N_REPS):
    torch.manual_seed(SEED + rep)
    model = DeepRNN(n_in, N_REC_RTRL, N_PATTERNS, n_layers=2)  # CPU intentional
    inputs, targets, mask = generate_batch(8, N_PATTERNS, 2, 1, 1, 'cpu')

    g_bptt = bptt_grads_deep(model, inputs, targets, mask)
    g_rtrl = compute_deep_rtrl_gradients(model, inputs, targets, mask, mse_error)

    for k in ['W_recs.0', 'W_recs.1', 'W_ffs.0', 'W_in', 'biases.0', 'biases.1']:
        v1, v2 = g_bptt[k].flatten(), g_rtrl[k].flatten()
        if v1.norm() < 1e-12: continue
        err = (v1/v1.norm() - v2/v2.norm()).norm().item()
        if err > max_dir_err:
            max_dir_err, worst_key = err, k

print(f"Max direction error over {N_REPS} trials: {max_dir_err:.2e}  (worst: {worst_key})")
status = 'PASS ✓' if max_dir_err < 1e-4 else 'FAIL ✗'
print(f"{status}: deep-RTRL {'matches' if max_dir_err < 1e-4 else 'deviates from'} BPTT")

---
## 5. Experiment 2B — Deep e-prop, 2 layers (Minimal Viable Result #3)

**Question:** How does gradient alignment change when credit must cross a layer boundary?

**Setup:** A 2-layer tanh RNN ($n=50$ per layer) on the store-and-recall task. We measure gradient cosine similarity for each layer group separately:

- **Layer 1** (deeper, closer to input): credit must travel through the feedforward Jacobian *and* be assigned temporally — two sources of approximation error
- **Layer 2** (output-adjacent): credit travels only one hop — less approximation error expected

**Part A — Cosine vs delay:** Sweep delays 1–20, report cosines for Layer 1, Layer 2, and all parameters combined.

**Part B — Learning curves:** Train all three rules at delay=2 to confirm the approximation quality translates to learning performance.

**Predictions:**
1. Layer 2 > Layer 1 in cosine similarity (output-adjacent parameters are easier to credit)
2. E-prop ≈ d=0 for both layers (carry still negligible for tanh)
3. Both approximate methods learn the task despite lower cosine similarity than single-layer

**How cross-layer traces work (see Fig 6 above):** Deep e-prop maintains a cross-layer eligibility trace $\varepsilon^{(2,1)}_t \in \mathbb{R}^{B \times n \times n \times n}$ for every pair $(l_\text{top}, l_\text{src})$. At each timestep, this trace is updated via the feedforward Jacobian (spatial) and the diagonal carry (temporal), then contracted with the learning signal to give the gradient for Layer 1's parameters.

In [ ]:
N_REC_2L   = 50
BATCH_2L   = BATCH_DEFAULT
delays_2l  = [1, 2, 3, 5, 10, 20]
n_trials   = 30

layer1_keys = ['W_recs.0', 'W_in', 'biases.0']
layer2_keys = ['W_recs.1', 'W_ffs.0', 'biases.1']
all_keys_2l = layer1_keys + layer2_keys

results_2l = {m: {g: [] for g in ['layer1','layer2','all']} for m in ['deep-eprop','d=0']}

for delay in delays_2l:
    sims = {m: {g: [] for g in ['layer1','layer2','all']} for m in ['deep-eprop','d=0']}
    for _ in range(n_trials):
        torch.manual_seed(np.random.randint(10000))
        model = DeepRNN(n_in, N_REC_2L, N_PATTERNS, n_layers=2).to(DEVICE)
        inp, tgt, msk = generate_batch(BATCH_2L, N_PATTERNS, delay, 1, 1, DEVICE)
        gb = bptt_grads_deep(model, inp, tgt, msk)
        ge = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=False)
        gd = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=True)
        for method, g in [('deep-eprop', ge), ('d=0', gd)]:
            sims[method]['layer1'].append(cosine_keys(g, gb, layer1_keys))
            sims[method]['layer2'].append(cosine_keys(g, gb, layer2_keys))
            sims[method]['all'].append(cosine_keys(g, gb, all_keys_2l))
    for m in ['deep-eprop','d=0']:
        for grp in ['layer1','layer2','all']:
            results_2l[m][grp].append(np.nanmean(sims[m][grp]))
    print(f"  delay={delay:3d}  "
          f"eprop(L1/L2)={results_2l['deep-eprop']['layer1'][-1]:.3f}/{results_2l['deep-eprop']['layer2'][-1]:.3f}  "
          f"d0(L1/L2)={results_2l['d=0']['layer1'][-1]:.3f}/{results_2l['d=0']['layer2'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, method, label in zip(axes, ['deep-eprop','d=0'], ['Deep e-prop','d=0']):
    ax.plot(delays_2l, results_2l[method]['layer1'], marker='s', linestyle='--', label='Layer 1 (deeper)')
    ax.plot(delays_2l, results_2l[method]['layer2'], marker='o', linestyle='-',  label='Layer 2 (output-adj.)')
    ax.plot(delays_2l, results_2l[method]['all'],    marker='^', linestyle=':',  label='All params', alpha=0.7)
    ax.axhline(0, color='gray', linestyle=':')
    ax.set_xlabel('Delay (steps)'); ax.set_ylabel('Gradient cosine similarity')
    ax.set_title(f'{label} — 2-layer network'); ax.legend(fontsize=8); ax.set_ylim(-0.1, 1.05)
fig.suptitle('Deep e-prop and d=0 gradient alignment with BPTT')
fig.tight_layout()
save_fig(fig, 'exp2b_2layer_tanh_store_recall_cosine_by_layer')
plt.show()

In [ ]:
N_STEPS_2L = 1000
LR_2L      = 1e-3
EVAL_2L    = 50
delay_2l   = 2

curves_2l = {}
for label, d_zero, use_bptt in [('deep-eprop', False, False), ('d=0', True, False), ('BPTT', False, True)]:
    torch.manual_seed(SEED)
    model = DeepRNN(n_in, N_REC_2L, N_PATTERNS, n_layers=2).to(DEVICE)
    accs = []
    for step in range(N_STEPS_2L):
        inp, tgt, msk = generate_batch(BATCH_2L, N_PATTERNS, delay_2l, 1, 1, DEVICE)
        if use_bptt:
            grads = bptt_grads_deep(model, inp, tgt, msk)
        else:
            grads = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=d_zero)
        apply_grads_deep(model, grads, LR_2L)
        if step % EVAL_2L == 0:
            with torch.no_grad(): out, _ = model(inp)
            accs.append(task_accuracy(out, tgt, msk))
    curves_2l[label] = accs
    print(f"  [{label}] final acc={accs[-1]:.3f}")

steps_2l = list(range(0, N_STEPS_2L, EVAL_2L))
fig, ax = plt.subplots(figsize=(7, 4))
for label, marker in [('deep-eprop','o'), ('d=0','s'), ('BPTT','^')]:
    ax.plot(steps_2l, curves_2l[label], label=label, marker=marker, markersize=3)
ax.axhline(1/N_PATTERNS, color='gray', linestyle='--', label='chance')
ax.set_xlabel('Training step'); ax.set_ylabel('Accuracy')
ax.set_title(f'Deep e-prop learning curves (2-layer, delay={delay_2l})')
ax.legend(); fig.tight_layout()
save_fig(fig, 'exp2b_2layer_tanh_store_recall_delay2_learning_curves')
plt.show()

---
## 6. Experiment 3 — Depth sweep L=1..5

**Question:** How does gradient alignment scale with network depth, and does it follow a regular pattern?

**Setup:** Networks of depth $L \in \{1, 2, 3, 4, 5\}$ ($n=50$ per layer) on the store-and-recall task. For each depth we:

1. **Measure gradient cosine similarity** on untrained models across delays $D \in \{2, 5, 10\}$
2. **Train all three rules** for 600 steps at delay=2 and measure final accuracy

**Why this matters:** Each additional layer introduces one more cross-layer trace approximation. If the errors compound multiplicatively, cosine should fall roughly geometrically with $L$. If errors are additive, the fall is more gradual.

**Memory note:** The cross-layer traces have shape $(B, n, n, n)$ per layer pair, scaling as $O(L^2 \cdot B \cdot n^3)$. At $L=5, n=50, B=32$: roughly 160 MB — manageable on a GPU.

**Runtime note:** E-prop's inner loop is inherently sequential (one timestep at a time), limiting GPU utilisation. Each layer $L$ requires all $L(L-1)/2$ cross-layer trace updates per step. Timing is printed next to each result.

**Predictions:**
1. Cosine decreases monotonically with $L$ — each layer adds error
2. The e-prop vs d=0 gap remains small (carry is still negligible for tanh)
3. All methods reach 100% accuracy at delay=2 regardless of depth (the task is short enough)
4. The per-layer cosine pattern from Experiment 2 (output-adjacent > deeper) holds at all depths

In [ ]:
# Depth sweep config — taken from MASTER CONFIG block above.
# DEPTHS_SWEEP, DELAYS_SWEEP, N_REC_DS, etc. already set.
# This cell is kept for readability; variables are already defined.
print(f"Depth sweep: depths={DEPTHS_SWEEP}, delays={DELAYS_SWEEP}, "
      f"n_rec={N_REC_DS}, {N_TRIALS_DS} trials, {N_STEPS_DS} steps")

In [ ]:
import time

ds_cosine = {m: {d: [] for d in DELAYS_SWEEP} for m in ['deep-eprop','d=0']}

for L in DEPTHS_SWEEP:
    t0 = time.time()
    for delay in DELAYS_SWEEP:
        se, sd = [], []
        for trial in range(N_TRIALS_DS):
            torch.manual_seed(SEED + trial * 100 + L * 10 + delay)
            model = DeepRNN(n_in, N_REC_DS, N_PATTERNS, n_layers=L).to(DEVICE)
            inp, tgt, msk = generate_batch(BATCH_DS, N_PATTERNS, delay, 1, 1, DEVICE)
            gb = bptt_grads_deep(model, inp, tgt, msk)
            ge = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=False)
            gd = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=True)
            all_k = list(model.state_dict().keys())
            se.append(cosine_keys(ge, gb, all_k))
            sd.append(cosine_keys(gd, gb, all_k))
        ds_cosine['deep-eprop'][delay].append(np.nanmean(se))
        ds_cosine['d=0'][delay].append(np.nanmean(sd))
    dt = time.time() - t0
    print(f"  L={L}  ({dt:.1f}s)  " + "  ".join(
        f"d={d}: eprop={ds_cosine['deep-eprop'][d][-1]:.3f} d0={ds_cosine['d=0'][d][-1]:.3f}"
        for d in DELAYS_SWEEP))
# ── Incremental save ─────────────────────────────────────────────────────────
import json as _json, os
os.makedirs("results", exist_ok=True)
with open("results/exp3_depth_sweep_cosine.json", "w") as _f:
    _json.dump({"depths": DEPTHS_SWEEP, "delays": DELAYS_SWEEP,
                "cosines": ds_cosine}, _f, default=float)
print("Saved exp3 cosine metrics.")

In [ ]:
colors_ds = plt.cm.viridis(np.linspace(0.15, 0.85, len(DELAYS_SWEEP)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, method, title in zip(axes, ['deep-eprop','d=0'], ['Deep e-prop','d=0']):
    for delay, c in zip(DELAYS_SWEEP, colors_ds):
        ax.plot(DEPTHS_SWEEP, ds_cosine[method][delay], marker='o', color=c, label=f'delay={delay}')
    ax.set_xlabel('Layers L'); ax.set_ylabel('Gradient cosine similarity')
    ax.set_title(title); ax.set_xticks(DEPTHS_SWEEP)
    ax.axhline(0, color='gray', linestyle=':'); ax.legend(fontsize=8); ax.set_ylim(-0.05, 1.05)
fig.suptitle('Gradient alignment vs depth — deep e-prop approximations')
fig.tight_layout()
save_fig(fig, 'exp3_depth_sweep_tanh_store_recall_cosine_vs_depth')
plt.show()

In [ ]:
ds_curves = {}
delay_lc  = 2

for method, d_zero, use_bptt in [('deep-eprop', False, False), ('d=0', True, False), ('BPTT', False, True)]:
    for L in DEPTHS_SWEEP:
        torch.manual_seed(SEED)
        model = DeepRNN(n_in, N_REC_DS, N_PATTERNS, n_layers=L).to(DEVICE)
        accs = []
        for step in range(N_STEPS_DS):
            inp, tgt, msk = generate_batch(BATCH_DS, N_PATTERNS, delay_lc, 1, 1, DEVICE)
            if use_bptt:
                grads = bptt_grads_deep(model, inp, tgt, msk)
            else:
                grads = compute_deep_eprop_gradients(model, inp, tgt, msk, mse_error, d_zero=d_zero)
            apply_grads_deep(model, grads, LR_DS)
            if step % EVAL_DS == 0:
                with torch.no_grad(): out, _ = model(inp)
                accs.append(task_accuracy(out, tgt, msk))
        ds_curves[(method, L)] = accs
        print(f"  [{method}, L={L}] final={accs[-1]:.3f}")

steps_ds = list(range(0, N_STEPS_DS, EVAL_DS))
depth_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(DEPTHS_SWEEP)))
method_labels = {'deep-eprop': 'Deep e-prop', 'd=0': 'd=0', 'BPTT': 'BPTT'}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, method in zip(axes, ['deep-eprop','d=0','BPTT']):
    for L, c in zip(DEPTHS_SWEEP, depth_colors):
        ax.plot(steps_ds, ds_curves[(method, L)], color=c, label=f'L={L}', marker='o', markersize=2.5)
    ax.axhline(1/N_PATTERNS, color='gray', linestyle='--', label='chance')
    ax.set_xlabel('Training step'); ax.set_ylabel('Accuracy')
    ax.set_title(method_labels[method]); ax.legend(fontsize=8); ax.set_ylim(-0.05, 1.05)
fig.suptitle('Learning curves by depth — store-and-recall (delay=2)')
fig.tight_layout()
save_fig(fig, 'exp3_depth_sweep_tanh_store_recall_delay2_learning_curves')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for method, marker in [('deep-eprop','o'), ('d=0','s'), ('BPTT','^')]:
    finals = [ds_curves[(method, L)][-1] for L in DEPTHS_SWEEP]
    ax.plot(DEPTHS_SWEEP, finals, marker=marker, label=method_labels[method])
ax.axhline(1/N_PATTERNS, color='gray', linestyle='--', label='chance')
ax.set_xlabel('Layers L'); ax.set_ylabel('Final accuracy')
ax.set_title('Final accuracy vs depth — store-and-recall (delay=2)')
ax.set_xticks(DEPTHS_SWEEP); ax.legend(); ax.set_ylim(-0.05, 1.05)
fig.tight_layout()
save_fig(fig, 'exp3_depth_sweep_tanh_store_recall_delay2_final_accuracy')
plt.show()

---
## 4. Experiment 4 — Leaky Integrator RNN: α sweep

**Question:** Can we construct a setting where the temporal trace in e-prop clearly matters, even for smooth (tanh-like) activations?

The standard tanh RNN has diagonal carry $c \approx \psi \cdot W_{\text{rec}}[i,i] \approx 0.005$, so the eligibility trace resets every step and e-prop $\approx$ d=0. The **leaky integrator RNN** adds an explicit decay:

$$h_t = (1-\alpha)\,h_{t-1} + \alpha\,\tanh(W_{\text{rec}}\,h_{t-1} + W_{\text{in}}\,x_t + b)$$

The diagonal Jacobian becomes $(1-\alpha) + \alpha\,\psi_t[i]\,W_{\text{rec}}[i,i]$, so the **dominant carry is $(1-\alpha)$** — a constant that grows as $\alpha$ shrinks. At $\alpha=0.1$ the trace survives $\approx 10$ steps; d=0 discards it entirely.

**Experiment:** Sweep $\alpha \in \{0.05,\,0.1,\,0.2,\,0.5,\,1.0\}$ at delay=5. For each $\alpha$:
1. Gradient cosine similarity of e-prop and d=0 vs BPTT (30 trials, untrained model)
2. Learning curves at $\alpha=0.1$ to confirm the cosine gap translates to task performance

**Predictions:**
- e-prop cosine stays high as $\alpha$ decreases (carry ≈ $1-\alpha$ keeps the trace alive)
- d=0 cosine drops steeply (no memory of past states)
- At $\alpha=1$ both are identical (recovers standard tanh result)


In [ ]:
N_REC_LK   = 100
ALPHAS_LK  = [0.05, 0.1, 0.2, 0.5, 1.0]
DELAY_LK   = 5
BATCH_LK   = 256            # explicit: larger batch → stable cosine estimate
N_TRIALS_LK = 30
n_in_lk    = N_PATTERNS + 2


In [ ]:
# Gradient cosine vs alpha — leaky integrator RNN
import time
cos_lk_ep, cos_lk_d0 = [], []

for alpha_val in ALPHAS_LK:
    t0 = time.time()
    se, sd = [], []
    for trial in range(N_TRIALS_LK):
        torch.manual_seed(SEED + trial * 7)
        model = LeakyRNN(n_in_lk, N_REC_LK, N_PATTERNS, alpha=alpha_val).to(DEVICE)
        inp, tgt, msk = generate_batch(BATCH_LK, N_PATTERNS, DELAY_LK, 1, 1, DEVICE)

        # BPTT reference
        for p in model.parameters():
            if p.grad is not None: p.grad.zero_()
        out, _ = model(inp)
        _mse_loss(out, tgt, msk).backward()
        g_bptt = {k: p.grad.clone() for k, p in model.named_parameters() if p.grad is not None}

        g_ep = compute_eprop_leaky_gradients(model, inp, tgt, msk, sl_mse, d_zero=False)
        g_d0 = compute_eprop_leaky_gradients(model, inp, tgt, msk, sl_mse, d_zero=True)

        keys = ['W_rec', 'W_in', 'b_rec']
        se.append(cosine_keys(g_ep, g_bptt, keys))
        sd.append(cosine_keys(g_d0, g_bptt, keys))

    cos_lk_ep.append(float(np.nanmean(se)))
    cos_lk_d0.append(float(np.nanmean(sd)))
    print(f'  alpha={alpha_val:.2f}  ({time.time()-t0:.1f}s)  '
          f'eprop={cos_lk_ep[-1]:.4f}  d0={cos_lk_d0[-1]:.4f}  '
          f'gap={cos_lk_ep[-1]-cos_lk_d0[-1]:+.4f}')

# ── Incremental save ─────────────────────────────────────────────────────────
import json as _json, os
os.makedirs("results", exist_ok=True)
with open("results/exp4_leaky_cosine_vs_alpha.json", "w") as _f:
    _json.dump({"alphas": ALPHAS_LK, "cos_ep": cos_lk_ep, "cos_d0": cos_lk_d0}, _f,
               default=float)
print("Saved exp4 cosine metrics.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ALPHAS_LK, cos_lk_ep, marker='o', label='e-prop vs BPTT', color='tab:blue')
ax.plot(ALPHAS_LK, cos_lk_d0, marker='s', label='d=0 vs BPTT',    color='tab:orange')
ax.axhline(0, color='gray', linestyle=':')
ax.set_xlabel(r'Leak rate $\alpha$')
ax.set_ylabel('Gradient cosine similarity')
ax.set_title(f'Leaky RNN: e-prop vs d=0 gradient alignment (delay={DELAY_LK})')
ax.legend()
fig.tight_layout()
save_fig(fig, 'exp4_leaky_rnn_alpha_sweep_cosine')
plt.show()


In [ ]:
# ── Leaky RNN learning curves: D=5 (control) vs D=20 (critical) ─────────────
# Left panel  D=5 < τ=10: all three methods converge.
# Right panel D=20 = 2τ:  d=0 (no trace) never achieves full accuracy;
#                          e-prop and BPTT converge (e-prop more slowly).
# Gradient clipping (max_norm=1.0) stabilises SGD uniformly across methods;
# it does not change gradient directions, so the directional comparison is fair.

BATCH_LK_LC  = 256    # local override — larger batch reduces noise
N_STEPS_LC   = 2000
EVAL_LC      = 100    # 20 evaluation points
MAX_NORM_LC  = 1.0    # global gradient-vector clip norm

def _apply_clipped(model, grads, lr, max_norm):
    vec   = torch.cat([g.flatten() for g in grads.values()])
    scale = min(1.0, max_norm / (vec.norm().item() + 1e-8))
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in grads:
                p.add_(grads[name], alpha=-lr * scale)

# Fixed evaluation batches — NOT resampled each step
torch.manual_seed(0)
_ev_i5,  _ev_t5,  _ev_m5  = generate_batch(1024, N_PATTERNS, 5,  1, 1, DEVICE)
_ev_i20, _ev_t20, _ev_m20 = generate_batch(1024, N_PATTERNS, 20, 1, 1, DEVICE)

fig_lk, axes_lk = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
curves_lk_all   = {}
_colors  = {'e-prop': 'tab:blue', 'd=0': 'tab:orange', 'BPTT': 'tab:green'}
_markers = {'e-prop': 'o', 'd=0': 's', 'BPTT': '^'}

for ax, delay_lk_lc, ev_i, ev_t, ev_m in [
        (axes_lk[0],  5, _ev_i5,  _ev_t5,  _ev_m5),
        (axes_lk[1], 20, _ev_i20, _ev_t20, _ev_m20)]:

    curves_lk_all[delay_lk_lc] = {}
    for label, d_zero, use_bptt in [('e-prop', False, False),
                                     ('d=0',   True,  False),
                                     ('BPTT',  False, True )]:
        torch.manual_seed(SEED)
        model = LeakyRNN(n_in_lk, N_REC_LK, N_PATTERNS, alpha=ALPHA_LC).to(DEVICE)
        accs  = []
        for step in range(N_STEPS_LC):
            inp, tgt, msk = generate_batch(BATCH_LK_LC, N_PATTERNS, delay_lk_lc, 1, 1, DEVICE)
            if use_bptt:
                grads = bptt_grads_deep(model, inp, tgt, msk, loss_fn=_xent_loss)
            else:
                grads = compute_eprop_leaky_gradients(
                    model, inp, tgt, msk, xent_error, d_zero=d_zero)
            _apply_clipped(model, grads, LR_LK, MAX_NORM_LC)
            if step % EVAL_LC == 0:
                with torch.no_grad():
                    out, _ = model(ev_i)
                accs.append(task_accuracy(out, ev_t, ev_m))
        curves_lk_all[delay_lk_lc][label] = accs
        print(f"  [D={delay_lk_lc}, {label}] final={accs[-1]:.3f}")

    _steps_lc = list(range(0, N_STEPS_LC, EVAL_LC))
    for lbl in ['BPTT', 'e-prop', 'd=0']:   # draw BPTT first (underneath)
        _a  = curves_lk_all[delay_lk_lc][lbl]
        _sm = np.array([np.mean(_a[max(0, i-2):i+3]) for i in range(len(_a))])
        ax.plot(_steps_lc, _a,  _markers[lbl], ms=3, alpha=0.20, color=_colors[lbl])
        ax.plot(_steps_lc, _sm, lw=2, label=lbl, color=_colors[lbl])
    ax.axhline(1 / N_PATTERNS, color='gray', ls='--', alpha=0.4, label='chance')
    ax.set_title(f'D={delay_lk_lc}  (α={ALPHA_LC}, τ≈{1/ALPHA_LC:.0f})', fontsize=12)
    ax.set_xlabel('Training step', fontsize=12)
    ax.set_ylim(-0.05, 1.08)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
    if ax is axes_lk[0]:
        ax.set_ylabel('Accuracy', fontsize=12)
        ax.legend(fontsize=10)

fig_lk.suptitle(f'Leaky RNN — {N_PATTERNS}-way store-and-recall (SGD+clip, LR={LR_LK})', fontsize=13)
fig_lk.tight_layout()
save_fig(fig_lk, 'exp4_leaky_learning_curves')
plt.show()

import json as _json
with open('results/exp4_leaky_curves.json', 'w') as _f:
    _json.dump({'steps': _steps_lc,
                'curves': {str(d): curves_lk_all[d] for d in curves_lk_all}}, _f)
print("Saved exp4 learning curves.")


---
## 5. Experiment 5 — Credit assignment across TIME and DEPTH, simultaneously

**Question.** Millidge (2025) shows deep e-prop *should* assign credit across both
depth and time. Does it, in practice, *at the same time*?

**Setup.** A 2-layer **leaky** `DeepRNN` with a FAST lower layer (transient feature
extractor, α=0.5) and a SLOW top layer (integrator, α=0.05), trained on a
hierarchical **classify-then-count** task (`tasks/hierarchical_cue.py`): each cue is
a mean-zero rising/falling temporal motif. The lower layer must learn the temporal
feature (**depth**); the top layer must accumulate the per-cue results across a
silent delay (**time**). Because the motifs are mean-zero, a frozen/random lower
layer (a reservoir) cannot fake the feature — so lower-layer credit genuinely matters.

**Two controls on the top-layer trace** `ε^z = (∂z/∂h)·ε^h + (∂z/∂z₍ₜ₋₁₎)·ε^z₍ₜ₋₁₎`:
- `ablate_spatial` (∂z/∂h = 0): removes the **depth** path → lower-layer grads → 0.
- `ablate_temporal` (∂z/∂z₍ₜ₋₁₎ = 0): removes the **cross-layer temporal** path.

**Third control — readout-only (reservoir)** (5.4): both recurrent layers stay
FROZEN at random init; only `W_out`/`b_out` train (the e-prop readout gradient is
exact). This is a 2-layer random reservoir + linear readout, and it is the natural
**floor** for `ablate_spatial`: that ablation freezes layer 0 but still trains
layer 1, so `ablate_spatial − readout-only` measures what training the top layer
buys over a random reservoir — i.e. whether depth credit is actually necessary.

**Predictions.** (5.1) full deep e-prop aligns with BPTT for *both* layers, while
each control breaks one dimension; most of the lower-layer credit flows through the
cross-layer temporal trace ε^z. (5.2) **BPTT ≥ full e-prop > both controls** in
learning. The full multi-seed study (incl. a delay sweep, E3) lives in
`experiments/deep_credit_time_depth.py`; the cells below run a compact version.

In [ ]:
# ── Experiment 5 setup ──────────────────────────────────────────────────────
from models.deep_rnn          import DeepRNN
from learning_rules.deep_eprop import compute_deep_eprop_gradients, xent_error
from learning_rules.bptt       import compute_bptt_gradients, _xent_loss
from learning_rules.interface  import apply_gradients
from tasks.hierarchical_cue    import generate_batch as hcue, task_accuracy as hacc
from utils                     import cosine_sim_grads, flat_grads
import time, json, warnings

E5_NREC   = 32
E5_ALPHA  = [0.5, 0.05]          # fast lower extractor + slow top integrator
E5_NCUES  = 3
E5_TASK   = dict(cue_duration=3, inter_cue_interval=2, amp=2.0, feature_noise=0.15)
E5_LOWER  = ["W_in", "W_recs.0", "biases.0"]      # lower-layer (layer-0) params
E5_UPPER  = ["W_recs.1", "W_ffs.0", "biases.1"]   # top-layer (layer-1) params
E5_DELAYS     = [6, 12, 20]
E5_DELAY_MAIN = 12               # learning-curve delay; also the summary-bar delay
E5_STEPS      = 2000             # ↑ (legacy fixed budget; superseded by E5_MAX_STEPS)
E5_SEEDS      = 8                # learning-curve seeds
E5_COS_SEEDS  = 16               # gradient-cosine seeds  ↑
E5_BATCH      = globals().get("BATCH_DEFAULT", 48)
E5_LR         = 1e-3             # Adam learning rate (5.2 trains with Adam — see below)
E5_EVAL       = 100
E5_EVAL_N     = 512              # samples per eval batch (×reps) — de-noises curves
E5_EVAL_REPS  = 4

# ── Optimiser + convergence-stopping config (used in 5.2) ────────────────────
# 5.2 trains with Adam (per-parameter step normalisation), NOT plain SGD. The credit
# ablations distort gradient *magnitude* far more than *direction* (ablate_temporal
# collapses the lower-layer gradient ~14x but keeps cosine≈0.98; ablate_spatial
# zeroes it). Under shared-LR SGD that magnitude deficit masquerades as a lower final
# accuracy; Adam removes magnitude as a variable per synapse (and is the optimiser
# used in the e-prop literature and our SHD scripts). With magnitude neutralised all
# rules converge to the SAME accuracy on this task — the credit-quality difference is
# in convergence SPEED, which the learning curves show. Each method trains to its own
# convergence (patience, with a min-steps floor so a slow starter isn't cut off early).
E5_GRAD_CLIP = 1.0               # global-norm gradient clip (matches SHD scripts)
E5_MIN_STEPS = 1500              # don't allow early stop before this (slow-starter guard)
E5_MAX_STEPS = 4000              # convergence cap / runaway guard
E5_PATIENCE  = 8                 # evals (×E5_EVAL steps) of no EMA improvement before stopping
E5_TOL       = 5e-3              # min EMA improvement that counts as progress
E5_TAIL_K    = 5                 # evals tail-averaged for the converged-accuracy readout

def e5_batch(B, delay, seed):
    inp, tgt, msk = hcue(B, n_cues=E5_NCUES, delay=delay, seed=seed, **E5_TASK)
    return inp.to(DEVICE), tgt.to(DEVICE), msk.to(DEVICE)

def e5_stats(res):
    # mean and standard error over seeds (rows); all-nan cols (e.g. the unrun tail of
    # an early-stopped seed) return nan/0 without noisy warnings.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mean = np.nanmean(res, axis=0); sd = np.nanstd(res, axis=0, ddof=1)
    n = np.sum(~np.isnan(res), axis=0)
    sem = np.where(n > 1, sd / np.sqrt(np.maximum(n, 1)), 0.0)
    return mean, sem

print("Exp 5 ready. alpha", E5_ALPHA, "| seeds: cosine", E5_COS_SEEDS, "learning", E5_SEEDS)


In [ ]:
# ── 5.1  Gradient credit: per-layer cosine vs BPTT + cross-temporal share ────
print(f"5.1 gradient credit ({E5_COS_SEEDS} seeds) ...")
E5_CK = ["full_low", "full_up", "temp_low", "temp_up", "spat_low", "spat_up", "xshare"]
e5_mean, e5_sem = {}, {}
for d in E5_DELAYS:
    rows = []
    for s in range(E5_COS_SEEDS):
        torch.manual_seed(1000 + s)
        m = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
        inp, tgt, msk = e5_batch(E5_BATCH, d, 5000 + s)
        gb = compute_bptt_gradients(m, inp, tgt, msk, _xent_loss)
        gf = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="full")
        gt = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="ablate_temporal")
        gs = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="ablate_spatial")
        a, b = flat_grads(gf, E5_LOWER), flat_grads(gt, E5_LOWER)
        rows.append([cosine_sim_grads(gf, gb, E5_LOWER), cosine_sim_grads(gf, gb, E5_UPPER),
                     cosine_sim_grads(gt, gb, E5_LOWER), cosine_sim_grads(gt, gb, E5_UPPER),
                     cosine_sim_grads(gs, gb, E5_LOWER), cosine_sim_grads(gs, gb, E5_UPPER),
                     ((a - b).norm() / (a.norm() + 1e-12)).item()])
    mu, er = e5_stats(np.array(rows, dtype=float))
    e5_mean[d] = dict(zip(E5_CK, mu)); e5_sem[d] = dict(zip(E5_CK, er))
    print(f"  D={d:3d}: full low={e5_mean[d]['full_low']:.3f}±{e5_sem[d]['full_low']:.3f} "
          f"top={e5_mean[d]['full_up']:.3f} | temporal low={e5_mean[d]['temp_low']:.3f}±{e5_sem[d]['temp_low']:.3f}"
          f" | spatial low=0 | xshare={e5_mean[d]['xshare']:.2f}")

def _ser(stat, k):
    return np.nan_to_num(np.array([stat[d][k] for d in E5_DELAYS]))

# Figure 1 — per-layer cosine vs delay, with stderr bands
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
for k, c, ls, lab, mfc in [("full_up", "C0", ":", "full · output-adjacent", "none"), ("full_low", "C0", "-", "full · input-adjacent", "C0"),
                      ("temp_low", "C3", "-", "ablate_temporal · input-adjacent", "C3"),
                      ("spat_low", "C1", "-", "ablate_spatial · input-adjacent (=0)", "C1")]:
    mu, er = _ser(e5_mean, k), _ser(e5_sem, k)
    ax.plot(E5_DELAYS, mu, marker="o", color=c, ls=ls, label=lab, markerfacecolor=mfc)
    ax.fill_between(E5_DELAYS, mu - er, mu + er, color=c, alpha=0.15)
ax.axhline(0, color="gray", ls=":"); ax.set_xlabel("delay D"); ax.set_ylabel("cosine vs BPTT")
ax.set_ylim(-0.1, 1.05); ax.legend(fontsize=8)
ax = axes[1]
mu, er = _ser(e5_mean, "xshare"), _ser(e5_sem, "xshare")
ax.plot(E5_DELAYS, mu, "o-", color="C2"); ax.fill_between(E5_DELAYS, mu - er, mu + er, color="C2", alpha=0.15)
ax.set_xlabel("delay D"); ax.set_ylabel("||full - ablate_temporal|| / ||full||"); ax.set_ylim(0, 1.05)
fig.tight_layout(); save_fig(fig, "exp5_gradient_credit"); plt.show()

# Figure 2 — summary bars at the main delay (reuses the same cosines)
db = E5_DELAY_MAIN if E5_DELAY_MAIN in e5_mean else E5_DELAYS[len(E5_DELAYS) // 2]
kf = {("full", "low"): "full_low", ("full", "top"): "full_up",
      ("ablate_temporal", "low"): "temp_low", ("ablate_temporal", "top"): "temp_up",
      ("ablate_spatial", "low"): "spat_low", ("ablate_spatial", "top"): "spat_up"}
fig, ax = plt.subplots(figsize=(7, 4.5)); x = np.arange(2); w = 0.26
for i, (meth, c) in enumerate([("full", "C0"), ("ablate_temporal", "C3"), ("ablate_spatial", "C1")]):
    mus = [np.nan_to_num(e5_mean[db][kf[(meth, l)]]) for l in ("low", "top")]
    ers = [np.nan_to_num(e5_sem[db][kf[(meth, l)]]) for l in ("low", "top")]
    ax.bar(x + (i - 1) * w, mus, w, yerr=ers, capsize=4, color=c, label=meth.replace("ablate_", "ablate "))
ax.set_xticks(x); ax.set_xticklabels(["input-adjacent", "output-adjacent"]); ax.set_ylim(-0.05, 1.05)
ax.axhline(0, color="gray", ls=":"); ax.set_ylabel("cosine vs BPTT")
ax.legend(fontsize=8); fig.tight_layout(); save_fig(fig, "exp5_credit_summary"); plt.show()

with open("results/exp5_gradient_credit.json", "w") as _f:
    json.dump({"delays": E5_DELAYS, "mean": e5_mean, "sem": e5_sem, "n": E5_COS_SEEDS}, _f)
print("Saved exp5 gradient credit + summary.")

In [ ]:
# ── 5.2  Adam learning curves: credit quality sets convergence SPEED ──────────
# Trained with Adam, which normalises the per-synapse step size and so removes the
# ablations' gradient-MAGNITUDE deficit, leaving only gradient direction/structure.
# Under fair optimisation every TRAINABLE rule converges to the SAME final accuracy
# on this task, so the credit-assignment difference shows up as convergence SPEED
# (full > ablate_temporal > ablate_spatial), which these curves display. Each method
# trains to its own convergence (patience + min-steps floor). There is no plateau-
# height claim among the trainable rules; the SPEED differences are significance-
# tested in 5.3. The readout-only RESERVOIR (both hidden layers frozen at random
# init; only W_out trains) is trained here too and overlaid as the FLOOR — it does
# NOT converge to the trainable rules' accuracy, so its gap to the ablation curves
# shows how much even a distorted credit signal buys over a pure random reservoir
# (quantified + significance-tested in 5.4).
print(f"5.2 learning curves (D={E5_DELAY_MAIN}, {E5_SEEDS} seeds, Adam lr={E5_LR:.0e}) ...")

def e5_eval(m, delay):
    accs = []
    for e in range(E5_EVAL_REPS):
        ie, te, me = e5_batch(E5_EVAL_N, delay, 90000 + e)
        with torch.no_grad():
            o, _ = m(ie)
        accs.append(hacc(o, te, me))
    return float(np.mean(accs))

e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
              ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]

def e5_grads(method, m, inp, tgt, msk):
    if method == "bptt":
        return compute_bptt_gradients(m, inp, tgt, msk, _xent_loss)
    if method == "readout":                       # readout-only reservoir control (5.4):
        g = compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode="full")
        return {k: v for k, v in g.items() if k in ("W_out", "b_out")}   # freeze BOTH layers
    return compute_deep_eprop_gradients(m, inp, tgt, msk, xent_error, mode=method)

# ── Patience-based training with Adam: stop once each method has converged ────
def e5_train(method, seed):
    torch.manual_seed(seed)
    m = DeepRNN(5, E5_NREC, 2, n_layers=2, alpha=E5_ALPHA).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=E5_LR)
    accs, ema, best, since, st = [], None, -1.0, 0, 0
    while True:
        if st % E5_EVAL == 0:
            a = e5_eval(m, E5_DELAY_MAIN); accs.append(a)
            ema = a if ema is None else 0.3 * a + 0.7 * ema
            if ema > best + E5_TOL:
                best, since = ema, 0
            else:
                since += 1
            if (since >= E5_PATIENCE and st >= E5_MIN_STEPS) or st >= E5_MAX_STEPS:
                break
        inp, tgt, msk = e5_batch(E5_BATCH, E5_DELAY_MAIN, 10_000 + seed * 1_000_000 + st)
        g = e5_grads(method, m, inp, tgt, msk)
        opt.zero_grad()
        for pname, p in m.named_parameters():            # custom grads -> Adam step
            p.grad = g.get(pname, torch.zeros_like(p))
        torch.nn.utils.clip_grad_norm_(m.parameters(), E5_GRAD_CLIP)
        opt.step()
        st += 1
    return accs            # ragged: length = #evals until this run converges

def e5_tail_slope(r):      # per-100-step slope over the converged tail (≈0 if flat)
    k = min(E5_TAIL_K, len(r))
    return float(np.polyfit(np.arange(k), np.asarray(r[-k:]), 1)[0]) if k >= 2 else 0.0

def e5_fit_curve(rows):    # ragged per-seed eval curves -> (mean, sem) on a nan-padded grid
    L = max(len(r) for r in rows)
    M = np.full((len(rows), L), np.nan)
    for i, r in enumerate(rows):
        M[i, :len(r)] = r
    return e5_stats(M)

e5_curves, e5_convacc, e5_stops, e5_rows = {}, {}, {}, {}
for meth, lab in e5_methods:
    t0 = time.time()
    rows = [e5_train(meth, s) for s in range(E5_SEEDS)]   # ragged per-seed eval curves
    e5_rows[meth] = rows
    e5_curves[meth]  = e5_fit_curve(rows)                 # nan-aware mean / SEM over seeds
    e5_convacc[meth] = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in rows])  # converged acc
    e5_stops[meth]   = [(len(r) - 1) * E5_EVAL for r in rows]
    slope = float(np.mean([e5_tail_slope(r) for r in rows]))
    print(f"  {lab:22s} conv-acc={e5_convacc[meth].mean():.3f}  "
          f"steps-to-converge≈{int(np.mean(e5_stops[meth]))}  tail-slope/100={slope:+.4f}  "
          f"[{time.time()-t0:.0f}s]")
print("Convergence check: conv-acc should be ~equal across the TRAINABLE rules and "
      "|tail-slope/100|≈0 (raise E5_MAX_STEPS if any still slopes up).")

# ── Readout-only RESERVOIR floor (both hidden layers frozen at random init) ───
# Same schedule/optimiser; kept OUT of e5_methods so the 5.3 speed stats stay on the
# rules that converge to equal accuracy. Overlaid on the figure as the floor and
# reused by 5.4 (so 5.4 does not retrain it).
t0 = time.time()
res_rows = [e5_train("readout", s) for s in range(E5_SEEDS)]
e5_rows["readout"]    = res_rows
e5_curves["readout"]  = e5_fit_curve(res_rows)
e5_convacc["readout"] = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in res_rows])
e5_stops["readout"]   = [(len(r) - 1) * E5_EVAL for r in res_rows]
print(f"  {'readout-only (reservoir)':22s} conv-acc={e5_convacc['readout'].mean():.3f}  "
      f"steps-to-converge≈{int(np.mean(e5_stops['readout']))}  [{time.time()-t0:.0f}s]  "
      f"(floor; excluded from the equal-accuracy / 5.3 speed comparison)")

# ── Figure: Adam curves — trainable rules converge equal (speed differs), ─────
#            with the random-reservoir floor overlaid.
col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1",
       "readout": "C7"}
e5_plot_methods = e5_methods + [("readout", "readout-only (reservoir)")]
fig, ax = plt.subplots(figsize=(7, 4.5))
for meth, lab in e5_plot_methods:
    mu, er = e5_curves[meth]
    x = np.arange(len(mu)) * E5_EVAL
    ls = "--o" if meth == "readout" else "-o"
    ax.plot(x, mu, ls, ms=3, color=col[meth], label=lab)
    ax.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.15)
ax.axhline(0.5, color="gray", ls="--", label="chance")
ax.set_xlabel("training step"); ax.set_ylabel("accuracy (held-out)"); ax.set_ylim(0.45, 1.02)
ax.set_title(f"Adam (lr={E5_LR:.0e}): trainable rules converge equal; reservoir floors below")
ax.legend(fontsize=8)
fig.tight_layout(); save_fig(fig, "exp5_learning_curves"); plt.show()

# Snapshot the prior shared-LR SGD curves once (before overwrite) for before/after contrast.
import os, shutil
_old, _snap = "results/exp5_learning_curves.json", "results/exp5_learning_curves_sgd.json"
if os.path.exists(_old) and not os.path.exists(_snap):
    shutil.copy(_old, _snap); print(f"Snapshotted prior SGD curves -> {_snap}")

with open(_old, "w") as _f:
    json.dump({"eval_every": E5_EVAL, "optimizer": "adam", "lr": float(E5_LR),
               "curves":      {m: [c[0].tolist(), c[1].tolist()] for m, c in e5_curves.items()},
               "conv_acc":    {m: e5_convacc[m].tolist()          for m, _ in e5_plot_methods},
               "stop_steps":  {m: e5_stops[m]                     for m, _ in e5_plot_methods},
               "per_seed":    {m: e5_rows[m]                      for m, _ in e5_plot_methods}}, _f)
print("Saved Adam exp5 learning curves incl. reservoir floor (+ per-seed curves for 5.3/5.4).")


In [ ]:
# ── 5.3  Speed significance: are the convergence-rate gaps real? ──────────────
# Adam makes every rule converge to the same accuracy, so the credit-assignment
# signal is in SPEED. Two independent, paired (per-seed) tests on the 5.2 curves —
# no extra training. Comparisons: full vs ablate_temporal, full vs ablate_spatial,
# ablate_temporal vs ablate_spatial (Holm-corrected over the 3).
from experiments.stats import paired_report, holm, cluster_perm_test

# Re-run-safe: rebuild per-seed curves from disk if 5.2 isn't in memory.
try:
    e5_rows; e5_methods; col; E5_EVAL
except NameError:
    with open("results/exp5_learning_curves.json") as _f:
        _d = json.load(_f)
    e5_methods = [("bptt", "BPTT"), ("full", "deep e-prop (full)"),
                  ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]
    e5_rows = {m: _d["per_seed"][m] for m, _ in e5_methods}
    E5_EVAL = _d["eval_every"]
    col = {"bptt": "k", "full": "C0", "ablate_temporal": "C3", "ablate_spatial": "C1"}

E5_SPEED_COMPS = [("full", "ablate_temporal", "full vs ablate_temporal", "C4"),
                  ("full", "ablate_spatial",  "full vs ablate_spatial",  "C5"),
                  ("ablate_temporal", "ablate_spatial", "temporal vs spatial", "C6")]
_lab = {m: lab for m, lab in e5_methods}

def _stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

# ── (A) Steps-to-threshold ────────────────────────────────────────────────────
def steps_to_threshold(r, theta, eval_every=E5_EVAL):
    """Interpolated training step at which curve r first reaches theta (NaN if never)."""
    r = np.asarray(r, float)
    above = np.where(r >= theta)[0]
    if len(above) == 0:
        return float("nan")
    i = int(above[0])
    if i == 0:
        return 0.0
    y0, y1 = r[i - 1], r[i]
    frac = 0.0 if y1 == y0 else (theta - y0) / (y1 - y0)
    return (i - 1 + min(max(frac, 0.0), 1.0)) * eval_every

def speed_threshold_report(theta):
    s2t = {m: np.array([steps_to_threshold(r, theta) for r in e5_rows[m]])
           for m, _ in e5_methods}
    cens = {m: int(np.isnan(s2t[m]).sum()) for m, _ in e5_methods}
    reports = [(lab, a, b, paired_report(s2t[a], s2t[b])) for a, b, lab, _ in E5_SPEED_COMPS]
    padj = holm([r["p_perm"] for *_, r in reports])
    print(f"\nSteps to reach acc≥{theta:.2f}  (paired sign-flip perm, Holm over {len(reports)}):")
    for m, _ in e5_methods:
        c = f"  ({cens[m]} censored)" if cens[m] else ""
        print(f"    {_lab[m]:22s} mean={np.nanmean(s2t[m]):6.0f} steps{c}")
    print(f"    {'comparison':26s} {'Δsteps':>7} {'dz':>6} {'perm p':>8} {'Holm':>8}")
    for (lab, a, b, r), pa in zip(reports, padj):
        print(f"    {lab:26s} {r['mean_diff']:+7.0f} {r['cohen_dz']:+6.2f} "
              f"{r['p_perm']:8.4f} {pa:8.4f}  {_stars(pa)}")
    return s2t, reports, padj

E5_THETA = 0.9
s2t, e5_spd_reports, e5_spd_padj = speed_threshold_report(E5_THETA)
speed_threshold_report(0.8)        # robustness check at a lower threshold

# Figure A (independent): mean steps-to-threshold per method ± SEM, with brackets.
order = ["bptt", "full", "ablate_temporal", "ablate_spatial"]
short = {"bptt": "BPTT", "full": "full", "ablate_temporal": "ablate\ntemporal",
         "ablate_spatial": "ablate\nspatial"}
means = {m: float(np.nanmean(s2t[m])) for m in order}
sems  = {m: float(np.nanstd(s2t[m], ddof=1) / np.sqrt(np.sum(~np.isnan(s2t[m])))) for m in order}
figA, axA = plt.subplots(figsize=(7, 4.6)); xs = np.arange(len(order))
axA.bar(xs, [means[m] for m in order], yerr=[sems[m] for m in order], capsize=4,
        color=[col[m] for m in order])
axA.set_xticks(xs); axA.set_xticklabels([short[m] for m in order])
axA.set_ylabel(f"training steps to reach acc ≥ {E5_THETA:.2f}")
axA.set_title(f"Convergence speed (Adam): steps to {E5_THETA:.0%} accuracy")
y0 = max(means[m] + sems[m] for m in order)
for k, (a, b, lab, _) in enumerate(E5_SPEED_COMPS):
    xa, xb = order.index(a), order.index(b); y = y0 * (1.06 + 0.10 * k)
    axA.plot([xa, xa, xb, xb], [y - y0 * 0.02, y, y, y - y0 * 0.02], color="k", lw=1)
    axA.text((xa + xb) / 2, y, _stars(e5_spd_padj[k]), ha="center", va="bottom", fontsize=11)
axA.set_ylim(0, y0 * 1.45)
figA.tight_layout(); save_fig(figA, "exp5_speed_threshold"); plt.show()

# ── (B) Cluster-permutation: where over training are the gaps significant? ─────
# Common rectangular window: up to the earliest stop across all methods/seeds.
n_common = min(len(r) for m, _ in e5_methods for r in e5_rows[m])
A = {m: np.array([r[:n_common] for r in e5_rows[m]]) for m, _ in e5_methods}
t_axis = np.arange(n_common) * E5_EVAL

print(f"\nCluster-permutation significant intervals (window 0–{t_axis[-1]} steps, "
      f"pointwise α=0.05, FWER-controlled over time):")
e5_sig_intervals = {}
for a, b, lab, _c in E5_SPEED_COMPS:
    cl = cluster_perm_test(A[a] - A[b])
    sig = [c for c in cl if c["p_cluster"] < 0.05]
    e5_sig_intervals[(a, b)] = sig
    spans = ", ".join(f"[{t_axis[c['t0']]}–{t_axis[c['t1']]}] steps (p={c['p_cluster']:.3f})"
                      for c in sig) or "none"
    print(f"    {lab:26s} {spans}")

# Figure B (independent): mean curves over the window + per-comparison sig bars.
figB, axB = plt.subplots(figsize=(7, 4.8))
for meth, lab in e5_methods:
    axB.plot(t_axis, A[meth].mean(0), "-o", ms=3, color=col[meth], label=lab)
axB.axhline(0.5, color="gray", ls="--", lw=0.8)
axB.set_xlabel("training step"); axB.set_ylabel("accuracy (held-out)")
axB.set_ylim(0.40, 1.02)
axB.set_title("Where convergence-speed gaps are significant (cluster permutation)")
ybar = 0.46
for k, (a, b, lab, cc) in enumerate(E5_SPEED_COMPS):
    y = ybar - 0.018 * k
    for c in e5_sig_intervals[(a, b)]:
        axB.plot([t_axis[c["t0"]], t_axis[c["t1"]]], [y, y], color=cc, lw=4, solid_capstyle="butt")
    axB.plot([], [], color=cc, lw=4, label=f"sig: {lab}")
axB.legend(fontsize=7, ncol=2, loc="lower right")
figB.tight_layout(); save_fig(figB, "exp5_speed_intervals"); plt.show()
print("Saved exp5_speed_threshold + exp5_speed_intervals.")


---
### 5.4 Readout-only reservoir control — is depth credit actually necessary?

A **third control**: freeze *both* recurrent layers at their random init and train
only the readout (a 2-layer random reservoir + linear readout / deep ESN). It is
the natural floor for the depth ablation:

- `ablate_spatial` — layer 0 frozen (grad = 0), **layer 1 trained**;
- `ablate_temporal` — cross-layer temporal carry off, **both layers trained**;
- `readout-only`  — layer 0 **and** layer 1 frozen.

The reservoir is trained on the same schedule in **5.2** and overlaid on the
learning-curve figure, so you can read the gap between each credit-carrying rule
(including *both* ablations) and the pure-reservoir floor directly off the curves.
Here we reuse those runs for the quantitative comparison:

- `ablate_spatial − readout-only` isolates *what training the top layer buys over a
  random reservoir*;
- `ablate_temporal − readout-only` isolates *what the surviving depth credit buys*.

If a rule ≫ reservoir, its learning-curve separation is a real credit effect; if
`ablate_spatial ≈ readout-only`, the trained top layer does not rescue the task from
a random lower layer and depth credit is essential. (Reuses 5.2 — run 5.2 first.)


In [ ]:
# ── 5.4  Readout-only reservoir control: is depth credit actually necessary? ──
# Both recurrent layers stay FROZEN at random init; only W_out/b_out train (the
# e-prop readout gradient is exact). This is a 2-layer random reservoir + linear
# readout — the natural FLOOR for ablate_spatial (which freezes layer 0 but TRAINS
# layer 1). (ablate_spatial - readout-only) = what training the top layer buys;
# (ablate_temporal - readout-only) = what the surviving depth credit buys.
# The reservoir is trained in 5.2 (overlaid on the learning-curve figure); here we
# reuse those runs and just do the quantitative comparison + paired significance.
print("5.4 readout-only reservoir control ...")

try:                    # reuses 5.2's per-seed converged accuracies + trainer
    e5_train; e5_convacc; e5_curves; e5_rows; col; E5_EVAL; E5_TAIL_K
except NameError:
    raise RuntimeError("Run 5.2 (the Adam learning-curve cell) before 5.4 — it "
                       "defines e5_train, e5_convacc, e5_curves and the reservoir run.")

# Reuse the reservoir trained in 5.2 (same schedule); fall back to training it here
# only if 5.2 predates this change and didn't produce it.
if "readout" in e5_convacc:
    res_convacc = e5_convacc["readout"]
    res_mu, res_sem = e5_curves["readout"]
    res_rows = e5_rows["readout"]
    print(f"  readout-only (reservoir) conv-acc={res_convacc.mean():.3f} (reused from 5.2)")
else:
    _t0 = time.time()
    res_rows = [e5_train("readout", s) for s in range(E5_SEEDS)]
    res_convacc = np.array([float(np.mean(r[-E5_TAIL_K:])) for r in res_rows])
    _Lr = max(len(r) for r in res_rows)
    _Mr = np.full((len(res_rows), _Lr), np.nan)
    for i, r in enumerate(res_rows):
        _Mr[i, :len(r)] = r
    res_mu, res_sem = e5_stats(_Mr)
    print(f"  readout-only (reservoir) conv-acc={res_convacc.mean():.3f}  [{time.time()-_t0:.0f}s]")
col.setdefault("readout", "C7")

# ── Converged-accuracy bars: trainable rules vs the reservoir floor ───────────
bar_methods = [("bptt", "BPTT"), ("full", "full"), ("ablate_temporal", "ablate\ntemporal"),
               ("ablate_spatial", "ablate\nspatial"), ("readout", "readout-only\n(reservoir)")]
acc_of = {m: (res_convacc if m == "readout" else e5_convacc[m]) for m, _ in bar_methods}
means = {m: float(np.mean(acc_of[m])) for m, _ in bar_methods}
sems  = {m: float(np.std(acc_of[m], ddof=1) / np.sqrt(len(acc_of[m]))) for m, _ in bar_methods}

figc, axc = plt.subplots(figsize=(7.5, 4.6)); xs = np.arange(len(bar_methods))
axc.bar(xs, [means[m] for m, _ in bar_methods], yerr=[sems[m] for m, _ in bar_methods],
        capsize=4, color=[col[m] for m, _ in bar_methods])
axc.set_xticks(xs); axc.set_xticklabels([lab for _, lab in bar_methods])
axc.set_ylabel("converged accuracy (held-out)"); axc.axhline(0.5, color="gray", ls="--", label="chance")
axc.set_ylim(0.45, 1.05); axc.legend(fontsize=8)
axc.set_title("Depth ablation vs a random reservoir: does training the top layer rescue it?")
figc.tight_layout(); save_fig(figc, "exp5_reservoir_control"); plt.show()

# ── Learning-curve overlay: reservoir floor against the trainable rules ───────
figd, axd = plt.subplots(figsize=(7, 4.5))
for meth, lab in [("bptt", "BPTT"), ("full", "full"),
                  ("ablate_temporal", "ablate temporal"), ("ablate_spatial", "ablate spatial")]:
    mu, er = e5_curves[meth]; x = np.arange(len(mu)) * E5_EVAL
    axd.plot(x, mu, "-o", ms=3, color=col[meth], label=lab)
    axd.fill_between(x, mu - er, mu + er, color=col[meth], alpha=0.12)
xr = np.arange(len(res_mu)) * E5_EVAL
axd.plot(xr, res_mu, "--o", ms=3, color=col["readout"], label="readout-only (reservoir)")
axd.fill_between(xr, res_mu - res_sem, res_mu + res_sem, color=col["readout"], alpha=0.12)
axd.axhline(0.5, color="gray", ls="--", label="chance")
axd.set_xlabel("training step"); axd.set_ylabel("accuracy (held-out)"); axd.set_ylim(0.45, 1.02)
axd.legend(fontsize=8); axd.set_title("Reservoir floor vs trainable-hidden rules")
figd.tight_layout(); save_fig(figd, "exp5_reservoir_curves"); plt.show()

# ── Paired significance: each trainable rule vs the reservoir floor ──────────
# Does every credit-carrying rule (incl. BOTH ablations) sit significantly above a
# pure random reservoir? full/ablate_temporal >> reservoir and ablate_spatial >>
# reservoir all indicate a real gap; ablate_spatial ~= reservoir would instead mean
# the frozen lower layer is not rescued by training the top (depth credit essential).
from experiments.stats import paired_report, holm
comps = [("full",            "full − reservoir"),
         ("ablate_temporal", "ablate_temporal − reservoir"),
         ("ablate_spatial",  "ablate_spatial − reservoir")]
reports = [(lab, paired_report(e5_convacc[m], res_convacc)) for m, lab in comps]
padj = holm([r["p_perm"] for _, r in reports])
print(f"  {'comparison':30s} {'meanD':>8} {'dz':>7} {'perm p':>8} {'Holm p':>8}  sig")
for (lab, r), pa in zip(reports, padj):
    sig = "***" if pa < 0.001 else "**" if pa < 0.01 else "*" if pa < 0.05 else "ns"
    print(f"  {lab:30s} {r['mean_diff']:+8.3f} {r['cohen_dz']:+7.2f} {r['p_perm']:8.4f} {pa:8.4f}  {sig}")

with open("results/exp5_reservoir_control.json", "w") as _f:
    json.dump({"delay": E5_DELAY_MAIN, "n_seeds": E5_SEEDS,
               "conv_acc": {**{m: acc_of[m].tolist() for m, _ in bar_methods if m != "readout"},
                            "readout": res_convacc.tolist()},
               "means": means, "sems": sems,
               "res_curve": [res_mu.tolist(), res_sem.tolist()],
               "comparisons": [{"comparison": lab, **r, "p_holm": float(pa)}
                               for (lab, r), pa in zip(reports, padj)],
               "per_seed_readout": res_rows}, _f)
print("Interpretation: every trainable rule (incl. both ablations) >> reservoir means")
print("even a distorted credit signal beats a pure random reservoir; the ablation↔reservoir")
print("gaps are the learning-curve separations you now see overlaid in 5.2.")
print("Saved exp5_reservoir_control + exp5_reservoir_curves.")


---
## 11. Summary and conclusions

### Results at a glance

| Experiment | Key finding | Confirmed? |
|-----------|-------------|:--:|
| **1 — Single-layer, tanh** | e-prop ≈ d=0 (gap < 0.004): the diagonal tanh carry is ≈ 0.005, so the trace barely persists; both reach high cosine with BPTT at short delay | ✅ |
| **2A — Deep-RTRL verification** | Deep-RTRL matches BPTT to numerical precision (direction error < 10⁻⁶) — the deep recursion is exact *before* the e-prop approximation | ✅ |
| **2B — 2-layer tanh** | Output-adjacent layer cosine ≈ 0.82, deeper layer ≈ 0.67; e-prop ≈ d=0 for vanilla tanh | ✅ |
| **3 — Depth sweep** | Gradient–BPTT cosine decreases monotonically with depth; e-prop ≈ d=0 throughout (vanilla tanh) | ✅ |
| **4 — Leaky RNN (α sweep)** | The e-prop > d=0 gap grows as α decreases (stronger leak, longer trace): e-prop captures the (1−α) temporal carry that d=0 discards | ✅ |
| **5 — Time + depth (main result)** | Full deep e-prop tracks BPTT for BOTH layers; `ablate_spatial`→lower-layer credit = 0, `ablate_temporal`→lower credit degraded (≈90% of it flows through the cross-layer trace εᶻ); learning: BPTT ≥ full e-prop > both controls, gap widening with delay | ✅ |

### Main conclusions

**1. For vanilla tanh RNNs, e-prop ≈ d=0.** The diagonal temporal carry (≈ 0.005/step) is so small that the eligibility trace resets almost immediately, so dropping it (the d=0 rule) barely changes the gradient. (Consistent with the Shalev-Merin 2026 analysis for randomly-initialised networks.)

**2. Leaky neurons create a meaningful temporal trace.** With carry ≈ (1−α)/step the trace survives ≈ 1/(1−α) steps, so e-prop assigns real credit across a silent delay; the e-prop vs d=0 gap grows with both delay and leak strength.

**3. Gradient alignment degrades with depth but stays positive.** Each extra layer adds one diagonal approximation in the spatial credit path, so cosine-with-BPTT decreases with depth — yet stays well above zero, enough to train.

**4. (Main result) Deep e-prop assigns credit across time AND depth simultaneously.** On a hierarchical *classify-then-count* task — the lower layer must learn a temporal feature (depth) whose per-cue output the top layer accumulates over a delay (time) — full deep e-prop aligns with BPTT for *both* layers and trains the network nearly as well as BPTT. The two controls on the top-layer trace `εᶻ = (∂z/∂h)·εʰ + (∂z/∂zₜ₋₁)·εᶻₜ₋₁` each break it: removing the **depth** path (∂z/∂h = 0) zeros the lower-layer gradient; removing the **cross-layer temporal** path (∂z/∂zₜ₋₁ = 0) degrades it (≈90% of lower-layer credit flows through `εᶻ`). Learning ordering is **BPTT ≥ full deep e-prop > both controls**, with the gap widening as the delay grows.

### Limitations and future work

- All experiments use **random initialisation**; after training, recurrent self-weights may grow and change the effective temporal carry.
- The exact deep e-prop cross-layer trace costs **O(L²·B·n³) memory**, which bounds the depth/width at which it is practical; Exp 5 uses a fixed 2-layer width and one task.
- BPTT saturates the Exp-5 task (≈100%); a harder task would further sharpen the BPTT-vs-e-prop gap.
- A commented **sMNIST appendix** (T=784) is included but not run by default — long sequences are where the tanh temporal approximation is expected to fail most.

All figures (PDF + SVG) are written to `results/` (wiped & regenerated on every run) and `figures/`. Run the cell below to push outputs to GitHub.

---
## Push results to GitHub

Run the cell below to commit all generated figures and push them to the repo.
You need a GitHub **Personal Access Token** with `repo` scope stored as a Colab secret
named `GITHUB_TOKEN` (Colab sidebar → 🔑 Secrets).

In [ ]:
import subprocess

# ── Configure git identity ───────────────────────────────────────────────────
subprocess.run(['git', 'config', 'user.name',  'Simon Peter'],        check=True)
subprocess.run(['git', 'config', 'user.email', 'go75wiv@mytum.de'],   check=True)

# ── Authenticate via GitHub token stored in Colab Secrets ───────────────────
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')   # set this in the 🔑 Secrets panel
subprocess.run([
    'git', 'remote', 'set-url', 'origin',
    f'https://{token}@github.com/simonpeter02/e-prop-in-deep-networks.git'
], check=True)

# ── Stage all generated figures ──────────────────────────────────────────────
subprocess.run(['git', 'add', '-A', 'results/', 'figures/'], check=True)  # -A stages deletions → results/ fully rewritten

# Check if there is anything new to commit
status = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True)
if not status.stdout.strip():
    print("Nothing new to commit — all figures already up to date.")
else:
    print("Staging:\n" + status.stdout)
    subprocess.run([
        'git', 'commit', '-m',
        'Add generated figures from Colab run\n\nCo-Authored-By: Claude Opus 4.8 (1M context) <noreply@anthropic.com>'
    ], check=True)
    subprocess.run(['git', 'push', 'origin', 'main'], check=True)
    print("\nPushed successfully.")

---
## Appendix — Sequential MNIST (commented out)

The cells below contain the sMNIST / psMNIST experiments from the original
notebook.  They are **not run by default** because T=784 steps through
online rules takes 60+ min on Colab CPU.

To run them, uncomment the cells and execute.  They use the `DeepRNN(n_layers=1)`
model on the 10-class MNIST classification problem.

Note: these cells require `torchvision` for MNIST download (auto-installed by pip).

In [ ]:
# from tasks.smnist          import generate_batch as smni_batch
# from tasks.smnist          import task_accuracy   as smni_acc
# from learning_rules.deep_eprop import xent_error
# from learning_rules.bptt       import _xent_loss
# 
# # Config
# N_REC_SMNI   = 64     # hidden units; keep small so each T=784 step is fast
# BATCH_SMNI   = 64     # smaller than BATCH_DEFAULT: each sample touches T=784 timesteps
# N_STEPS_SMNI = 500
# LR_SMNI      = 1e-3
# EVAL_SMNI    = 50
# N_COS_SMNI   = 10     # trials for gradient cosine estimate
# 
# n_in_smni  = 1        # one pixel per timestep
# n_out_smni = 10       # 10-class digit classification
# 
# # Sanity check — also triggers MNIST download
# inp, tgt, msk = smni_batch(4, device=DEVICE)
# print(f"sMNIST batch shape:  inputs {tuple(inp.shape)},  "
#       f"targets {tuple(tgt.shape)},  mask sum {int(msk.sum())}")
# print(f"Chance accuracy: {1/n_out_smni:.2f}")
# 
# # Helper: BPTT with cross-entropy loss (works for VanillaRNN and DeepRNN)
# def bptt_xent(model, inputs, targets, mask):
#     for p in model.parameters():
#         if p.grad is not None: p.grad.zero_()
#     outputs, _ = model(inputs)
#     _xent_loss(outputs, targets, mask).backward()
#     return {k: p.grad.clone()
#             for k, p in model.named_parameters() if p.grad is not None}

In [ ]:
# # Gradient cosine on sMNIST (L=1) — compare to store-and-recall T≈4
# # Each trial: forward e-prop + d=0 + BPTT through T=784 steps; ~2-5 s/trial on GPU.
# print(f"Gradient cosine on sMNIST (T=784, L=1), {N_COS_SMNI} trials ...")
# sims_se, sims_sd = [], []
# 
# for trial in range(N_COS_SMNI):
#     torch.manual_seed(SEED + trial * 13)
#     model = DeepRNN(n_in_smni, N_REC_SMNI, n_out_smni, n_layers=1).to(DEVICE)
#     inp, tgt, msk = smni_batch(BATCH_SMNI, device=DEVICE)
#     gb = bptt_xent(model, inp, tgt, msk)
#     ge = compute_deep_eprop_gradients(model, inp, tgt, msk, xent_error, d_zero=False)
#     gd = compute_deep_eprop_gradients(model, inp, tgt, msk, xent_error, d_zero=True)
#     all_k = list(model.state_dict().keys())
#     sims_se.append(cosine_keys(ge, gb, all_k))
#     sims_sd.append(cosine_keys(gd, gb, all_k))
#     print(f"  trial {trial+1:2d}/{N_COS_SMNI}  eprop={sims_se[-1]:.3f}  d0={sims_sd[-1]:.3f}")
# 
# cos_smni_e = float(np.nanmean(sims_se))
# cos_smni_d = float(np.nanmean(sims_sd))
# print(f"\nsMNIST  (T=784):             eprop={cos_smni_e:.3f}  d0={cos_smni_d:.3f}")
# print(f"Store-and-recall (T≈4, d=2): eprop≈0.89    d0≈0.88")
# print(f"  Δ cosine due to longer T:  {cos_smni_e - 0.886:+.3f}")

In [ ]:
# # sMNIST learning curves
# # Estimated runtime: 5-15 min on GPU, 60+ min on CPU.
# print("Training on sMNIST (L=1, n=64, 500 steps) ...")
# curves_smni = {}
# 
# for label, d_zero, use_bptt in [('e-prop', False, False),
#                                   ('d=0',   True,  False),
#                                   ('BPTT',  False, True )]:
#     torch.manual_seed(SEED)
#     model = DeepRNN(n_in_smni, N_REC_SMNI, n_out_smni, n_layers=1).to(DEVICE)
#     accs  = []
#     for step in range(N_STEPS_SMNI):
#         inp, tgt, msk = smni_batch(BATCH_SMNI, device=DEVICE)
#         if use_bptt:
#             grads = bptt_xent(model, inp, tgt, msk)
#         else:
#             grads = compute_deep_eprop_gradients(
#                 model, inp, tgt, msk, xent_error, d_zero=d_zero)
#         apply_grads_deep(model, grads, LR_SMNI)
#         if step % EVAL_SMNI == 0:
#             with torch.no_grad():
#                 inp_e, tgt_e, msk_e = smni_batch(256, device=DEVICE, train=False)
#                 out_e, _ = model(inp_e)
#             accs.append(smni_acc(out_e, tgt_e, msk_e))
#             if step % 200 == 0:
#                 print(f"  [{label}] step {step:4d}  acc={accs[-1]:.3f}")
#     curves_smni[label] = accs
#     print(f"  [{label}] final acc={accs[-1]:.3f}\n")

In [ ]:
# # psMNIST learning curves (same setup, pixels permuted)
# print("Training on psMNIST (permuted pixels, L=1, n=64, 500 steps) ...")
# curves_psmni = {}
# 
# for label, d_zero, use_bptt in [('e-prop', False, False),
#                                   ('d=0',   True,  False),
#                                   ('BPTT',  False, True )]:
#     torch.manual_seed(SEED)
#     model = DeepRNN(n_in_smni, N_REC_SMNI, n_out_smni, n_layers=1).to(DEVICE)
#     accs  = []
#     for step in range(N_STEPS_SMNI):
#         inp, tgt, msk = smni_batch(BATCH_SMNI, permuted=True, device=DEVICE)
#         if use_bptt:
#             grads = bptt_xent(model, inp, tgt, msk)
#         else:
#             grads = compute_deep_eprop_gradients(
#                 model, inp, tgt, msk, xent_error, d_zero=d_zero)
#         apply_grads_deep(model, grads, LR_SMNI)
#         if step % EVAL_SMNI == 0:
#             with torch.no_grad():
#                 inp_e, tgt_e, msk_e = smni_batch(256, permuted=True, device=DEVICE, train=False)
#                 out_e, _ = model(inp_e)
#             accs.append(smni_acc(out_e, tgt_e, msk_e))
#             if step % 200 == 0:
#                 print(f"  [{label}] step {step:4d}  acc={accs[-1]:.3f}")
#     curves_psmni[label] = accs
#     print(f"  [{label}] final acc={accs[-1]:.3f}\n")

In [ ]:
# steps_smni  = list(range(0, N_STEPS_SMNI, EVAL_SMNI))
# colors_m    = {'e-prop': 'tab:blue', 'd=0': 'tab:orange', 'BPTT': 'tab:green'}
# markers_m   = {'e-prop': 'o',        'd=0': 's',           'BPTT': '^'}
# 
# fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
# for ax, curves, title in [(axes[0], curves_smni,  'sMNIST  (sequential pixels)'),
#                            (axes[1], curves_psmni, 'psMNIST (permuted pixels)')]:
#     for label in ['e-prop', 'd=0', 'BPTT']:
#         ax.plot(steps_smni, curves[label],
#                 label=label, color=colors_m[label],
#                 marker=markers_m[label], markersize=3)
#     ax.axhline(1/n_out_smni, color='gray', linestyle='--', label='chance')
#     ax.set_xlabel('Training step')
#     ax.set_ylabel('Test accuracy')
#     ax.set_title(title)
#     ax.legend()
#     ax.set_ylim(-0.02, 1.05)
# 
# fig.suptitle('sMNIST and psMNIST — learning curves (single-layer RNN, n=64, T=784)')
# fig.tight_layout()
# save_fig(fig, 'exp7_tanh_smnist_pmnist_t784_learning_curves')
# plt.show()
# 
# print("\n=== Final test accuracy (step 500) ===")
# print(f"{'Method':8s}  {'sMNIST':>8s}  {'psMNIST':>8s}")
# for label in ['e-prop', 'd=0', 'BPTT']:
#     print(f"{label:8s}  {curves_smni[label][-1]:>8.3f}  {curves_psmni[label][-1]:>8.3f}")
# print(f"\nGradient cosine (sMNIST, T=784, L=1): "
#       f"e-prop={cos_smni_e:.3f}  d0={cos_smni_d:.3f}")